# Roof Inspector dataset and training

Planned workflow only. Do not run these cells until you are ready to download images and train a new model.

Run this notebook from the `Roof_Inspector` project root so `data/` and `model.pkl` land in the right place.

Images found through DuckDuckGo stay copyrighted by their owners. Use them for local training. Do not publish the downloaded set.

The CatOrDog `model.pkl` is not used. This notebook trains a new ResNet-18 and exports a new file.


In [ ]:
from pathlib import Path

from ddgs import DDGS
from fastai.vision.all import (
    ImageDataLoaders,
    Resize,
    download_images,
    error_rate,
    get_image_files,
    resnet18,
    verify_images,
    vision_learner,
)

ROOT = Path('.').resolve()
DATA = ROOT / 'data'
RESULTS_PER_QUERY = 60

QUERIES = {
    'potential_repair_needed': [
        'damaged asphalt shingle roof',
        'missing shingles on roof',
        'roof covered with blue tarp',
    ],
    'no_obvious_repair_needed': [
        'new asphalt shingle roof',
        'intact residential shingle roof',
        'clean asphalt roof shingles',
    ],
}

assert DATA.exists(), 'Run this notebook from the Roof_Inspector project root.'


## Search and download

Each query asks DuckDuckGo for about 60 images. URLs are deduplicated within a class before download, so the same photo returned by more than one query is saved once.


In [ ]:
from ddgs.exceptions import DDGSException


def search_image_urls(query, max_results=RESULTS_PER_QUERY):
    results = DDGS().images(query=query, max_results=max_results)
    urls = []
    for item in results:
        url = item.get('image') if isinstance(item, dict) else None
        if url:
            urls.append(url)
    return urls


def unique_urls(urls):
    seen = set()
    ordered = []
    for url in urls:
        if url in seen:
            continue
        seen.add(url)
        ordered.append(url)
    return ordered


for label, queries in QUERIES.items():
    dest = DATA / label
    dest.mkdir(parents=True, exist_ok=True)
    found = []
    for query in queries:
        try:
            batch = search_image_urls(query)
        except DDGSException as exc:
            print(f'{label!r} / {query!r}: search failed ({type(exc).__name__}): {exc}')
            continue
        print(f'{label!r} / {query!r}: {len(batch)} urls')
        found.extend(batch)
    urls = unique_urls(found)
    print(f'{label}: {len(urls)} unique urls')
    download_images(dest, urls=urls)


## Remove broken downloads

`verify_images` drops files that are not readable images. After this cell, open both class folders and delete obvious misses yourself: interiors, diagrams, people, tools, logos, and photos that are not an asphalt shingle roof. Training should wait until that pass is done.


In [ ]:
failed = verify_images(get_image_files(DATA))
failed.map(Path.unlink)
print(f'Removed {len(failed)} unreadable files')

for label in QUERIES:
    count = len(get_image_files(DATA / label))
    print(f'{label}: {count} images remaining')


## Manual cleanup

Review `data/potential_repair_needed/` and `data/no_obvious_repair_needed/` before training. Delete irrelevant images, then rerun the count cell above. A first local experiment is useful around 80–120 kept photos per class.


## Train a new model

Folder names are the class labels. This starts from ImageNet weights inside `vision_learner`, not from the CatOrDog pet model.


In [ ]:
dls = ImageDataLoaders.from_folder(
    'data', valid_pct=0.2, seed=42, item_tfms=Resize(192)
)
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(3)


In [ ]:
learn.export('model.pkl')
print('Exported', ROOT / 'model.pkl')
print('Classes:', list(learn.dls.vocab))
